# Методы кластеризации — Задание

**Уровень:** для начинающих · **Библиотеки:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`, `scipy`

Выполняйте задания по порядку. Где написано `# ВАШ КОД`, нужно дописать свой код. После каждого задания есть ячейка проверки: если всё сделано верно, она выведет ✅.

## Содержание
1. Создаём и рассматриваем данные
2. Метод K-Means
3. Выбор числа кластеров: метод локтя
4. Коэффициент силуэта
5. Масштабирование признаков
6. Иерархическая кластеризация и дендрограмма
7. DBSCAN: кластеры сложной формы и шум
8. Реальные данные: ирисы Фишера
9. (Бонус) Сегментация клиентов
10. Вопросы для самопроверки

## Подготовка
Запустите ячейку, чтобы подключить библиотеки.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, make_moons, load_iris
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage

plt.rcParams["figure.figsize"] = (6, 4)
RANDOM_STATE = 42  # фиксируем случайность, чтобы результаты повторялись

---
## Задание 1. Создаём и рассматриваем данные

**Кластеризация** — это задача обучения *без учителя*: у нас нет правильных ответов, и алгоритм сам ищет группы похожих объектов.

Чтобы потренироваться, создадим искусственные данные, в которых группы заранее известны.

1. С помощью `make_blobs` создайте `300` точек с `4` центрами, `cluster_std=0.8`, `random_state=RANDOM_STATE`. Результат сохраните в `X` (признаки) и `y_true` (настоящие номера групп — мы будем использовать их только для проверки).
2. Выведите форму `X`.
3. Постройте диаграмму рассеяния `X` **без раскраски** — так данные выглядят для алгоритма, который не знает ответов.

💡 *Подсказка:* `plt.scatter(X[:, 0], X[:, 1])`

In [ ]:
X, y_true = None, None  # ВАШ КОД: make_blobs(...)

# ВАШ КОД: выведите форму X

# ВАШ КОД: постройте диаграмму рассеяния


In [ ]:
assert X.shape == (300, 2), "Ожидается X формы (300, 2)"
assert len(np.unique(y_true)) == 4, "Должно быть 4 истинные группы"
print("✅ Задание 1 выполнено")

---
## Задание 2. Метод K-Means

**K-Means** делит данные на `k` кластеров: выбирает `k` центров, относит каждую точку к ближайшему центру, пересчитывает центры как средние точек кластера — и так повторяет, пока центры не перестанут двигаться.

1. Создайте `KMeans` с `n_clusters=4`, `n_init=10`, `random_state=RANDOM_STATE`.
2. Обучите модель и получите метки кластеров: `labels = model.fit_predict(X)`.
3. Сохраните координаты центров в `centers` (атрибут `cluster_centers_`).
4. Постройте график: точки раскрасьте по `labels` (`c=labels`), центры отметьте красными крестиками (`marker="X"`, `s=200`, `c="red"`).

In [ ]:
kmeans = None   # ВАШ КОД: создайте KMeans
labels = None   # ВАШ КОД: обучите и получите метки
centers = None  # ВАШ КОД: центры кластеров

# ВАШ КОД: постройте график


In [ ]:
assert labels.shape == (300,), "labels должен содержать 300 значений"
assert centers.shape == (4, 2), "centers должен быть формы (4, 2)"
assert len(set(labels)) == 4, "Должно получиться 4 кластера"
print("✅ Задание 2 выполнено")

---
## Задание 3. Как выбрать число кластеров: метод локтя

В реальных задачах число кластеров заранее неизвестно. Один из способов его подобрать — **метод локтя**.

У обученного `KMeans` есть атрибут `inertia_` — сумма квадратов расстояний от точек до центров своих кластеров. Чем больше кластеров, тем меньше inertia, но после «правильного» `k` падение резко замедляется — на графике появляется «локоть».

1. В цикле для `k` от `1` до `10` обучите `KMeans` и сохраните `inertia_` в список `inertias`.
2. Постройте график зависимости `inertia` от `k` (`marker="o"`).
3. Ответьте в ячейке ниже: где находится «локоть»?

In [ ]:
inertias = []

# ВАШ КОД: цикл по k от 1 до 10

# ВАШ КОД: график


**Ваш ответ:** «Локоть» находится при k = ...

In [ ]:
assert len(inertias) == 10, "Нужно 10 значений inertia (k = 1..10)"
assert all(np.diff(inertias) < 0), "Inertia должна убывать с ростом k"
print("✅ Задание 3 выполнено")

---
## Задание 4. Коэффициент силуэта

Метод локтя не всегда даёт однозначный ответ. Вторая метрика — **коэффициент силуэта** (`silhouette_score`). Он принимает значения от −1 до 1: чем ближе к 1, тем лучше точки сидят «в своём» кластере и отделены от чужих.

1. Для `k` от `2` до `8` (для `k = 1` силуэт не определён) обучите `KMeans`, получите метки и посчитайте `silhouette_score(X, labels)`. Значения сохраните в список `sil_scores`.
2. Найдите `k` с максимальным силуэтом и сохраните в `best_k`.
3. Постройте график силуэта от `k`.

💡 *Подсказка:* `k_values[np.argmax(sil_scores)]`

### 📐 Как считается коэффициент силуэта

Силуэт сначала считается **для каждой точки** `i`, а потом усредняется.

- **a(i)** — среднее расстояние от точки `i` до *остальных точек своего кластера*. Показывает, насколько точка «своя» в кластере (чем меньше, тем лучше).
- **b(i)** — среднее расстояние от точки `i` до точек *ближайшего чужого кластера*. Для каждого другого кластера считается среднее расстояние, берётся наименьшее (чем больше, тем лучше).

$$s(i) = \frac{b(i) - a(i)}{\max\big(a(i),\, b(i)\big)}$$

Итоговый `silhouette_score` — это среднее значение $s(i)$ по всем точкам.

| $s(i)$ | Что значит |
|---|---|
| близко к **1** | $a \ll b$: точка плотно сидит в своём кластере и далека от чужих |
| около **0** | $a \approx b$: точка на границе двух кластеров |
| **< 0** | $a > b$: точка, скорее всего, попала не в тот кластер |

**Важно знать:**
- Для расчёта не нужны настоящие метки — это *внутренняя* метрика, она оценивает только геометрию кластеров.
- Если в кластере всего одна точка, то по соглашению $s(i) = 0$.
- Нужны попарные расстояния, поэтому на больших данных расчёт медленный (у `silhouette_score` есть параметр `sample_size`).

Ниже — расчёт «вручную» на маленьком примере и сверка с `sklearn`.

In [ ]:
from scipy.spatial.distance import cdist

# Маленький пример: 6 точек на плоскости, 2 кластера
P = np.array([[1, 1], [2, 1], [1, 2],    # кластер 0
              [6, 5], [7, 5], [6, 6]])   # кластер 1
lab = np.array([0, 0, 0, 1, 1, 1])

D = cdist(P, P)  # матрица попарных расстояний

def silhouette_of_point(i):
    own = (lab == lab[i]) & (np.arange(len(P)) != i)                 # свои, кроме самой точки
    a = D[i, own].mean()                                             # a(i)
    b = min(D[i, lab == c].mean() for c in set(lab) if c != lab[i])  # b(i)
    return (b - a) / max(a, b)

s_all = [silhouette_of_point(i) for i in range(len(P))]
print("s(i) по точкам:", np.round(s_all, 3))
print("Среднее (вручную):         ", round(float(np.mean(s_all)), 4))
print("silhouette_score (sklearn):", round(float(silhouette_score(P, lab)), 4))

In [ ]:
k_values = list(range(2, 9))
sil_scores = []

# ВАШ КОД: цикл по k

best_k = None  # ВАШ КОД: k с максимальным силуэтом
print("Лучшее k:", best_k)

# ВАШ КОД: график


In [ ]:
assert len(sil_scores) == 7, "Нужно 7 значений (k = 2..8)"
assert best_k == 4, "Для этих данных лучшее k должно быть равно 4"
print("✅ Задание 4 выполнено")

---
## Задание 5. Зачем нужно масштабирование признаков

K-Means опирается на **расстояния**. Если один признак измеряется в тысячах, а другой — в единицах, то расстояние определяется почти только первым.

Ниже создан набор данных из 3 групп клиентов. Оба признака помогают отличить группы, но `score` измеряется в единицах, а `income` — в десятках тысяч, поэтому без масштабирования `income` «перетягивает» на себя всё расстояние.

1. Обучите `KMeans(n_clusters=3)` на **исходных** данных, посчитайте `adjusted_rand_score(group, labels_raw)` → `ari_raw`.
2. Масштабируйте данные с помощью `StandardScaler().fit_transform(...)`, обучите `KMeans` заново → `ari_scaled`.
3. Сравните результаты. ARI = 1 означает идеальное совпадение с настоящими группами, ≈ 0 — случайное разбиение.

### 📐 Как считается ARI (Adjusted Rand Index)

**Задача:** сравнить два разбиения одних и тех же объектов — «настоящее» и найденное алгоритмом. Номера кластеров при этом не важны: разбиения `[0, 0, 1, 1]` и `[1, 1, 0, 0]` — одно и то же.

**Шаг 1. Индекс Рэнда (Rand Index).** Рассматриваем все пары объектов (их $\binom{n}{2}$). Пара «согласована», если оба разбиения относятся к ней одинаково: либо **в обоих** пара лежит в одном кластере, либо **в обоих** — в разных. Индекс Рэнда — доля согласованных пар.

**Шаг 2. Поправка на случайность.** Даже два случайных разбиения дают заметный Rand Index (много пар «в разных кластерах» совпадёт случайно). Поэтому из него вычитают ожидаемое значение при случайных разбиениях:

$$ARI = \frac{RI - E[RI]}{\max(RI) - E[RI]}$$

**Как считают на практике.** Строят таблицу сопряжённости: $n_{ij}$ — сколько объектов попало одновременно в истинную группу $i$ и найденный кластер $j$; $a_i$ — суммы по строкам, $b_j$ — суммы по столбцам. Тогда

$$ARI = \frac{\sum_{ij}\binom{n_{ij}}{2} - \dfrac{\sum_i\binom{a_i}{2}\,\sum_j\binom{b_j}{2}}{\binom{n}{2}}}{\dfrac{1}{2}\Big[\sum_i\binom{a_i}{2} + \sum_j\binom{b_j}{2}\Big] - \dfrac{\sum_i\binom{a_i}{2}\,\sum_j\binom{b_j}{2}}{\binom{n}{2}}}$$

| ARI | Что значит |
|---|---|
| **1** | разбиения совпадают (с точностью до переименования кластеров) |
| около **0** | результат не лучше случайного |
| **< 0** | хуже случайного (встречается редко) |

**Важно знать:** для ARI нужны настоящие метки — это *внешняя* метрика. В реальных задачах без разметки её посчитать нельзя, поэтому там опираются на силуэт и интерпретацию кластеров.

Ниже — расчёт «вручную» по этой формуле и сверка с `sklearn`.

In [ ]:
from math import comb

y_a = [0, 0, 0, 1, 1, 1, 2, 2]  # «настоящие» группы
y_b = [1, 1, 0, 0, 0, 2, 2, 2]  # результат кластеризации

table = pd.crosstab(np.array(y_a), np.array(y_b))   # таблица сопряжённости n_ij
display(table)

n = int(table.values.sum())
sum_ij = sum(comb(int(x), 2) for x in table.values.ravel())   # Σ C(n_ij, 2)
sum_a = sum(comb(int(x), 2) for x in table.sum(axis=1))       # Σ C(a_i, 2)
sum_b = sum(comb(int(x), 2) for x in table.sum(axis=0))       # Σ C(b_j, 2)

expected = sum_a * sum_b / comb(n, 2)      # ожидаемое значение при случайных разбиениях
max_index = (sum_a + sum_b) / 2
ari_manual = (sum_ij - expected) / (max_index - expected)

print("ARI (вручную):      ", round(ari_manual, 4))
print("adjusted_rand_score:", round(adjusted_rand_score(y_a, y_b), 4))

# Переименование кластеров ничего не меняет:
print("ARI([0,0,1,1], [1,1,0,0]) =", adjusted_rand_score([0, 0, 1, 1], [1, 1, 0, 0]))

*Данные для задания (уже написано за вас):*

In [ ]:
rng = np.random.RandomState(RANDOM_STATE)
group = rng.randint(0, 3, size=300)                    # настоящие группы
score = group * 3 + rng.normal(0, 0.7, size=300)               # маленький масштаб (единицы)
income = 40000 + group * 10000 + rng.normal(0, 6000, size=300)  # большой масштаб (десятки тысяч)
df_scale = pd.DataFrame({"score": score, "income": income})
df_scale.describe().round(1)

In [ ]:
# 1. Без масштабирования
labels_raw = None  # ВАШ КОД
ari_raw = None     # ВАШ КОД

# 2. С масштабированием
X_scaled = None       # ВАШ КОД
labels_scaled = None  # ВАШ КОД
ari_scaled = None     # ВАШ КОД

print(f"ARI без масштабирования: {ari_raw:.3f}")
print(f"ARI с масштабированием:  {ari_scaled:.3f}")

In [ ]:
assert ari_scaled > ari_raw, "После масштабирования качество должно вырасти"
print("✅ Задание 5 выполнено")

---
## Задание 6. Иерархическая кластеризация и дендрограмма

**Иерархическая (агломеративная) кластеризация** начинает с того, что каждая точка — отдельный кластер, и на каждом шаге объединяет две ближайшие группы. Результат можно изобразить **дендрограммой** — деревом объединений.

1. Возьмите первые 40 точек `X[:40]`, вычислите `Z = linkage(X[:40], method="ward")` и постройте `dendrogram(Z)`.
2. Обучите `AgglomerativeClustering(n_clusters=4, linkage="ward")` на **всех** данных `X`, получите `agg_labels`.
3. Посчитайте `ari_agg = adjusted_rand_score(y_true, agg_labels)`.

💡 *Подсказка:* чем выше «перемычка» на дендрограмме, тем дальше друг от друга были объединяемые кластеры.

In [ ]:
# 1. Дендрограмма для первых 40 точек
Z = None  # ВАШ КОД
# ВАШ КОД: dendrogram(Z) и подписи осей

# 2. Агломеративная кластеризация
agg_labels = None  # ВАШ КОД

# 3. Качество
ari_agg = None  # ВАШ КОД
print("ARI:", ari_agg)

In [ ]:
assert agg_labels.shape == (300,)
assert ari_agg > 0.8, "Для этих данных ARI должен быть высоким"
print("✅ Задание 6 выполнено")

---
## Задание 7. DBSCAN: кластеры сложной формы и шум

K-Means ищет «круглые» кластеры. **DBSCAN** находит группы по **плотности** точек, умеет находить кластеры произвольной формы и помечает выбросы меткой `-1` (шум). Число кластеров задавать не нужно, но есть два параметра:

- `eps` — радиус окрестности точки;
- `min_samples` — сколько соседей нужно, чтобы точка считалась «плотной».

Ниже созданы данные в форме двух полумесяцев.

1. Выполните `KMeans(n_clusters=2)` → `km_labels`.
2. Выполните `DBSCAN(eps=0.2, min_samples=5)` → `db_labels`.
3. Посчитайте `n_noise` — количество точек с меткой `-1`.
4. Посчитайте ARI обоих методов относительно `y_moons` (`ari_kmeans`, `ari_dbscan`).
5. Постройте два графика рядом (`plt.subplots(1, 2, figsize=(11, 4))`).
6. *(Необязательно)* В цикле переберите `eps` из `[0.1, 0.15, 0.2, 0.3]` и для каждого выведите число найденных кластеров и число шумовых точек. Как влияет `eps`?

*Данные для задания (уже написано за вас):*

In [ ]:
X_moons, y_moons = make_moons(n_samples=300, noise=0.06, random_state=RANDOM_STATE)
plt.scatter(X_moons[:, 0], X_moons[:, 1], s=30)
plt.title("Данные «две луны»")
plt.show()

In [ ]:
km_labels = None  # ВАШ КОД
db_labels = None  # ВАШ КОД

n_noise = None      # ВАШ КОД: число точек с меткой -1
ari_kmeans = None   # ВАШ КОД
ari_dbscan = None   # ВАШ КОД
print(f"Шумовых точек: {n_noise}")
print(f"ARI K-Means: {ari_kmeans:.3f}, ARI DBSCAN: {ari_dbscan:.3f}")

# ВАШ КОД: два графика рядом

# ВАШ КОД (необязательно): цикл по eps


**Вопрос:** почему K-Means не справился с этими данными? **Ваш ответ:** ...

In [ ]:
assert ari_dbscan > ari_kmeans, "DBSCAN должен лучше справиться с формой «полумесяцы»"
print("✅ Задание 7 выполнено")

---
## Задание 8. Реальные данные: ирисы Фишера

Применим кластеризацию к классическому набору данных — 150 цветков ириса с 4 измерениями (длина/ширина чашелистика и лепестка). В нём три вида ириса, но мы притворимся, что не знаем их, а потом сравним результат с правдой.

1. Загрузите данные: `iris = load_iris()`, признаки — `iris.data`, настоящие виды — `iris.target`.
2. Масштабируйте признаки (`StandardScaler`) → `X_iris`.
3. Выполните `KMeans(n_clusters=3)` → `iris_labels`.
4. Посчитайте `ari_iris` и постройте таблицу сопряжённости `pd.crosstab(iris.target, iris_labels)`.
5. Сожмите данные до 2 компонент с помощью `PCA(n_components=2)` → `X_pca` и покажите на графике: слева — кластеры, справа — настоящие виды.

In [ ]:
iris = load_iris()

X_iris = None       # ВАШ КОД: масштабированные признаки
iris_labels = None  # ВАШ КОД: KMeans, k = 3
ari_iris = None     # ВАШ КОД
print("ARI:", ari_iris)

# ВАШ КОД: таблица сопряжённости

# ВАШ КОД: PCA и два графика


In [ ]:
assert ari_iris > 0.5, "ARI должен быть выше 0.5"
print("✅ Задание 8 выполнено")

---
## Задание 9 (бонус). Сегментация клиентов магазина

Финальная мини-задача — как кластеризация применяется в бизнесе. Ниже создана таблица из 200 клиентов с двумя признаками: годовой доход (тыс. €) и «индекс трат» (1–100).

1. Масштабируйте признаки → `X_cust`.
2. Подберите `k` из диапазона `2–6` по коэффициенту силуэта → `best_k_cust`.
3. Обучите `KMeans` с этим `k` и добавьте столбец `cluster` в `customers`.
4. Посчитайте средние значения признаков и размер каждого кластера: `customers.groupby("cluster").agg(...)`.
5. Опишите словами, что за группы клиентов у вас получились, и придумайте каждой название.

*Данные для задания (уже написано за вас):*

In [ ]:
rng = np.random.RandomState(0)

def make_group(n, income_mean, spend_mean):
    return pd.DataFrame({
        "annual_income": rng.normal(income_mean, 6, n),
        "spending_score": np.clip(rng.normal(spend_mean, 8, n), 1, 100),
    })

customers = pd.concat([
    make_group(50, 25, 25),
    make_group(50, 25, 75),
    make_group(50, 85, 25),
    make_group(50, 85, 75),
], ignore_index=True)
customers.head()

In [ ]:
X_cust = None       # ВАШ КОД: масштабирование

best_k_cust = None  # ВАШ КОД: подбор k по силуэту

# ВАШ КОД: обучите KMeans с best_k_cust и добавьте столбец "cluster"

# ВАШ КОД: сводная таблица по кластерам


**Ваши названия сегментов:**

- Кластер 0: ...
- Кластер 1: ...
- Кластер 2: ...
- Кластер 3: ...

In [ ]:
assert "cluster" in customers.columns, "Добавьте столбец cluster"
assert best_k_cust == 4, "В данных заложено 4 группы клиентов"
print("✅ Задание 9 выполнено")

---
## Вопросы для самопроверки
Ответьте на вопросы своими словами.

**1. Чем кластеризация отличается от классификации?**

*Ваш ответ:* ...

**2. Что показывает inertia в K-Means и почему по ней нельзя выбирать k, просто беря минимум?**

*Ваш ответ:* ...

**3. Какие параметры нужно задать в K-Means, а какие — в DBSCAN?**

*Ваш ответ:* ...

**4. Что означает метка -1 в результатах DBSCAN?**

*Ваш ответ:* ...

**5. Что означает значение коэффициента силуэта, близкое к 0 или отрицательное?**

*Ваш ответ:* ...

---
## Шпаргалка: какой метод выбрать?

| Метод | Нужно задавать | Форма кластеров | Выбросы | Когда использовать |
|---|---|---|---|---|
| **K-Means** | число кластеров `k` | компактные, «круглые» | чувствителен | быстрый первый вариант, большие данные |
| **Иерархическая** | `k` или высота разреза | зависит от `linkage` | чувствительна | небольшие данные, нужна дендрограмма |
| **DBSCAN** | `eps`, `min_samples` | произвольная | находит и помечает как -1 | сложные формы, есть шум |

**Общий порядок работы:** очистить данные → масштабировать признаки → подобрать параметры (локоть, силуэт) → обучить → **интерпретировать** кластеры.